# Reproduce QEDCode's smoke run

This notebook repeats the **smoke run** from the [results page](https://github.com/AdityaTummuri/QEDCode/blob/main/docs/results.md):
Qwen3.5 4B on Benchy's `benchy-core` suite, once **bare** (one prompt, one answer) and once **through
QEDCode**, judged by hidden tests neither side sees.

It needs a free notebook GPU (Kaggle or Colab, when one is available) and downloads about 4 GB:
Ollama, the model and the Python packages. Everything stays inside this notebook's machine.
Nothing is uploaded; the report is printed at the end.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No NVIDIA GPU: switch the runtime to a GPU first.'

## 2. Install Ollama and the model (about 3.4 GB)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen3.5:4b

## 3. Install QEDCode and Benchy

In [ ]:
!pip install -q uv
!uv venv -q /content/qed --python 3.12 2>/dev/null || uv venv -q ./qed --python 3.12
import os, pathlib
VENV = pathlib.Path("/content/qed") if pathlib.Path("/content/qed").exists() else pathlib.Path("qed").resolve()
!uv pip install -q --python {VENV}/bin/python qedcode qedcode-benchy
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
!qed --version && benchy --version

## 4. Tell QEDCode to use the Ollama model

The same as choosing "use a model I already have" in `qed setup`.

In [ ]:
home = pathlib.Path.home() / ".qedcode"
home.mkdir(exist_ok=True)
(home / "config.toml").write_text('''[engine]
mode = "local"
speed = "normal"

[local]
backend = "ollama"
model = "qwen3.5:4b"
ctx = 16384
''')
!qed doctor || true

## 5. Run the smoke run (3 repetitions)

About 20–40 minutes on a T4. Each task runs bare and through QEDCode in a random order.

In [ ]:
!benchy run --model ollama:qwen3.5:4b --suite benchy-core --reps 3

## 6. The report

In [ ]:
!benchy report

Compare your numbers with the [results page](https://github.com/AdityaTummuri/QEDCode/blob/main/docs/results.md).
To share yours, run `benchy share` to see exactly what a contribution would contain, then
`benchy share --export` and open a pull request on
[QEDCode-benchy](https://github.com/AdityaTummuri/QEDCode-benchy) (it shows your GitHub account).